# Batch Suite2p Processing

This notebook batch-processes folders containing TIFF files with Suite2p. It mirrors the workflow from `GCaMP-analysis/preprocessing/batch_s2p.py`: discover TIFF folders under one or more roots, skip folders that already have completed Suite2p output, validate TIFF structure before running, remove incomplete binary/cache files, and run Suite2p folder-by-folder.

Edit the configuration cell below, then run the notebook cells in order.

In [1]:
from __future__ import annotations

from pathlib import Path
import copy
import traceback

import numpy as np
import tifffile

from suite2p import run_s2p
from suite2p.parameters import (
    SETTINGS,
    default_db,
    default_settings,
)


pynwb not installed, save_nwb, read_nwb, and nwb_to_binary will not work. Install with: pip install pynwb


In [2]:
settings_path = r"C:\Users\mzinn1\.suite2p\settings\settings_user.npy"
settings = np.load(settings_path, allow_pickle=True).item()
for key, value in settings.items():
    if isinstance(value, dict):
        print(f"{key}:")
        for subkey, subvalue in value.items():
            if isinstance(subvalue, dict):
                print(f"  {subkey}:")
                for subsubkey, subsubvalue in subvalue.items():
                    print(f"    {subsubkey}: {subsubvalue}")
            else:
                print(f"  {subkey}: {subvalue}")
    else:
        print(f"{key}: {value}")

torch_device: cuda
tau: 1.0
fs: 15.0
diameter: [32.0, 32.0]
run:
  do_registration: 1
  do_regmetrics: True
  do_detection: True
  do_deconvolution: True
  multiplane_parallel: False
io:
  combined: True
  save_mat: False
  save_NWB: False
  save_ops_orig: True
  delete_bin: True
  move_bin: False
registration:
  align_by_chan2: False
  nimg_init: 400
  maxregshift: 0.1
  do_bidiphase: False
  bidiphase: 0.0
  batch_size: 500
  nonrigid: True
  maxregshiftNR: 5
  block_size: [128.0, 128.0]
  smooth_sigma_time: 0.0
  smooth_sigma: 5.0
  spatial_taper: 40.0
  th_badframes: 1.0
  norm_frames: True
  snr_thresh: 1.2
  subpixel: 10
  two_step_registration: False
  reg_tif: False
  reg_tif_chan2: False
  upsample_meanImg: None
detection:
  algorithm: cellpose
  denoise: False
  block_size: [64.0, 64.0]
  nbins: 5000
  bin_size: None
  highpass_time: 100
  threshold_scaling: 0.2
  npix_norm_min: 0.0
  npix_norm_max: 5.0
  max_overlap: 0.15
  soma_crop: True
  chan2_threshold: 0.25
  cellpose_

In [3]:

# ---------------------------------------------------------------------
# Edit these paths and options before running the batch cell.
# ---------------------------------------------------------------------
# Native (new-format) nested settings saved by the current Suite2p GUI.
GUI_SETTINGS_PATH = Path.home() / ".suite2p" / "settings" / "settings_user.npy"

ROOTS = [
    Path(r"C:\Users\mzinn1\Desktop\DoD WT Cohorts\Ex Vivo Ca Imaging 710"),
]

SKIP_IF_PROCESSED = False
FRESH_REPROCESS = True  # Rebuild bad existing Suite2p outputs from the TIFFs.
FORCE_GPU = True
NCHANNELS = 1
FUNCTIONAL_CHAN = 1  # Suite2p uses 1-based channel numbering.
RUN_REGISTRATION = False  # Stationary ex vivo stacks; registration corrupts these data.

# Fallback cell diameter (pixels) for Cellpose, used only if the loaded settings


# Overrides applied on top of the loaded settings. Use None to leave unchanged.
# Any key from SETTINGS_KEY_PATHS below can be used (flat name). For the two
# ambiguous names ("batch_size", "block_size") use a dotted path to disambiguate,
# e.g. "detection.block_size" or "extraction.batch_size".
OVERRIDES = {
    # Ex vivo acquisition and segmentation settings. Everything else comes
    # from the usual Suite2p GUI settings loaded above.
    "fs": 3.0,
    "algorithm": "cellpose",
    "cellpose_model": "cpdino_ExVivoGCaMP",
}

# Every Suite2p setting keyed by its (flat) name -> location in the nested
# settings dict. Used by apply_overrides to route OVERRIDES to the right place.
# NOTE: "batch_size" and "block_size" exist in two sections; the entries below
# point at the registration copies. Use a dotted key in OVERRIDES to target the
# other one (e.g. "detection.block_size", "extraction.batch_size").
SETTINGS_KEY_PATHS = {
    "active_percentile": ("detection", "sparsery_settings", "active_percentile"),
    "algorithm": ("detection", "algorithm"),
    "align_by_chan2": ("registration", "align_by_chan2"),
    "allow_overlap": ("extraction", "allow_overlap"),
    "baseline": ("dcnv_preprocess", "baseline"),
    "batch_size": ("registration", "batch_size"),
    "bidiphase": ("registration", "bidiphase"),
    "bin_size": ("detection", "bin_size"),
    "block_size": ("registration", "block_size"),
    "cellpose_chan2": ("detection", "cellpose_chan2"),
    "cellpose_model": ("detection", "cellpose_settings", "cellpose_model"),
    "cellprob_threshold": ("detection", "cellpose_settings", "cellprob_threshold"),
    "chan2_threshold": ("detection", "chan2_threshold"),
    "circular_neuropil": ("extraction", "circular_neuropil"),
    "classifier_path": ("classification", "classifier_path"),
    "combined": ("io", "combined"),
    "connected": ("detection", "sourcery_settings", "connected"),
    "delete_bin": ("io", "delete_bin"),
    "denoise": ("detection", "denoise"),
    "diameter": ("diameter",),
    "do_bidiphase": ("registration", "do_bidiphase"),
    "do_deconvolution": ("run", "do_deconvolution"),
    "do_detection": ("run", "do_detection"),
    "do_registration": ("run", "do_registration"),
    "do_regmetrics": ("run", "do_regmetrics"),
    "flow_threshold": ("detection", "cellpose_settings", "flow_threshold"),
    "fs": ("fs",),
    "highpass_neuropil": ("detection", "sparsery_settings", "highpass_neuropil"),
    "highpass_spatial": ("detection", "cellpose_settings", "highpass_spatial"),
    "highpass_time": ("detection", "highpass_time"),
    "img": ("detection", "cellpose_settings", "img"),
    "inner_neuropil_radius": ("extraction", "inner_neuropil_radius"),
    "lam_percentile": ("extraction", "lam_percentile"),
    "max_ROIs": ("detection", "sparsery_settings", "max_ROIs"),
    "max_iterations": ("detection", "sourcery_settings", "max_iterations"),
    "max_overlap": ("detection", "max_overlap"),
    "maxregshift": ("registration", "maxregshift"),
    "maxregshiftNR": ("registration", "maxregshiftNR"),
    "min_neuropil_pixels": ("extraction", "min_neuropil_pixels"),
    "move_bin": ("io", "move_bin"),
    "multiplane_parallel": ("run", "multiplane_parallel"),
    "nbins": ("detection", "nbins"),
    "neuropil_coefficient": ("extraction", "neuropil_coefficient"),
    "neuropil_extract": ("extraction", "neuropil_extract"),
    "nimg_init": ("registration", "nimg_init"),
    "nonrigid": ("registration", "nonrigid"),
    "norm_frames": ("registration", "norm_frames"),
    "npix_norm_max": ("detection", "npix_norm_max"),
    "npix_norm_min": ("detection", "npix_norm_min"),
    "params": ("detection", "cellpose_settings", "params"),
    "params_chan2": ("detection", "cellpose_settings", "params_chan2"),
    "prctile_baseline": ("dcnv_preprocess", "prctile_baseline"),
    "preclassify": ("classification", "preclassify"),
    "reg_tif": ("registration", "reg_tif"),
    "reg_tif_chan2": ("registration", "reg_tif_chan2"),
    "save_NWB": ("io", "save_NWB"),
    "save_mat": ("io", "save_mat"),
    "save_ops_orig": ("io", "save_ops_orig"),
    "sig_baseline": ("dcnv_preprocess", "sig_baseline"),
    "smooth_masks": ("detection", "sourcery_settings", "smooth_masks"),
    "smooth_sigma": ("registration", "smooth_sigma"),
    "smooth_sigma_time": ("registration", "smooth_sigma_time"),
    "snr_thresh": ("registration", "snr_thresh"),
    "snr_threshold": ("extraction", "snr_threshold"),
    "soma_crop": ("detection", "soma_crop"),
    "spatial_scale": ("detection", "sparsery_settings", "spatial_scale"),
    "spatial_taper": ("registration", "spatial_taper"),
    "subpixel": ("registration", "subpixel"),
    "tau": ("tau",),
    "th_badframes": ("registration", "th_badframes"),
    "threshold_scaling": ("detection", "threshold_scaling"),
    "torch_device": ("torch_device",),
    "two_step_registration": ("registration", "two_step_registration"),
    "upsample_meanImg": ("registration", "upsample_meanImg"),
    "use_builtin_classifier": ("classification", "use_builtin_classifier"),
    "win_baseline": ("dcnv_preprocess", "win_baseline"),
}


In [4]:
class ContiguousImageJStack:
    def __init__(self, path: str):
        self.array = tifffile.memmap(path, mode="r")


def patch_suite2p_tiff_reader() -> None:
    """Patch Suite2p to read contiguous single-page ImageJ stacks as frame stacks."""
    from suite2p.io import tiff as suite2p_tiff

    if getattr(suite2p_tiff.open_tiff, "_batch_notebook_patched", False):
        return

    original_open_tiff = suite2p_tiff.open_tiff
    original_read_tiff = suite2p_tiff.read_tiff

    def open_tiff(path: str, sktiff: bool):
        with tifffile.TiffFile(path) as tif:
            series = tif.series[0]
            is_contiguous_imagej_stack = (
                len(tif.pages) == 1
                and len(series.shape) == 3
                and series.axes.endswith("YX")
                and series.shape[0] > 1
                and tif.pages[0].is_contiguous
            )

        if is_contiguous_imagej_stack:
            stack = ContiguousImageJStack(path)
            print(f"Reading contiguous ImageJ TIFF series as {stack.array.shape[0]} frames")
            return stack, stack.array.shape[0]

        return original_open_tiff(path, sktiff)

    def read_tiff(file, tif, Ltif, ix, batch_size, use_sktiff):
        if not isinstance(tif, ContiguousImageJStack):
            return original_read_tiff(file, tif, Ltif, ix, batch_size, use_sktiff)

        if ix >= Ltif:
            return None

        nfr = min(Ltif - ix, batch_size)
        images = np.asarray(tif.array[ix:ix + nfr])
        if images.dtype.type in (np.uint16, np.int32):
            images = (images // 2).astype(np.int16)
        elif images.dtype.type != np.int16:
            images = images.astype(np.int16)
        return images

    open_tiff._batch_notebook_patched = True
    suite2p_tiff.open_tiff = open_tiff
    suite2p_tiff.read_tiff = read_tiff


def find_tiffs(folder: Path) -> list[Path]:
    return sorted(
        path for path in folder.iterdir()
        if path.is_file() and path.suffix.lower() in {".tif", ".tiff"}
    )


def is_already_processed(folder: Path) -> bool:
    plane0 = folder / "suite2p" / "plane0"
    required_outputs = ("F.npy", "iscell.npy", "ops.npy")
    return all((plane0 / filename).is_file() for filename in required_outputs)


def remove_incomplete_cache(folder: Path, force: bool = False) -> list[Path]:
    if is_already_processed(folder) and not force:
        return []

    plane0 = folder / "suite2p" / "plane0"
    removed = []
    # Remove every artifact that can make Suite2p reuse stale channel or
    # registration state after an interrupted/failed run. This is especially
    # important when nchannels or functional_chan changed between attempts.
    for filename in (
        "data.bin", "data_chan2.bin",
        "data_raw.bin", "data_raw_chan2.bin",
        "db.npy", "settings.npy", "reg_outputs.npy",
        "detect_outputs.npy", "stat.npy", "ops.npy",
        "F.npy", "Fneu.npy", "F_chan2.npy", "Fneu_chan2.npy",
        "spks.npy", "iscell.npy", "redcell.npy",
    ):
        path = plane0 / filename
        if path.is_file():
            path.unlink()
            removed.append(path)
    return removed


def validate_tiff(path: Path) -> None:
    file_size = path.stat().st_size
    if file_size <= 16:
        raise ValueError(f"file is only {file_size} bytes")

    try:
        with tifffile.TiffFile(path) as tif:
            if len(tif.pages) == 0:
                raise ValueError("contains no TIFF pages")

            for page_number, page in enumerate(tif.pages):
                if not page.shape or any(size <= 0 for size in page.shape):
                    raise ValueError(f"page {page_number} has no image shape")

                if not page.dataoffsets or not page.databytecounts:
                    raise ValueError(f"page {page_number} has no pixel data")

                if len(page.dataoffsets) != len(page.databytecounts):
                    raise ValueError(f"page {page_number} has inconsistent pixel data offsets")

                for offset, byte_count in zip(page.dataoffsets, page.databytecounts):
                    if byte_count <= 0 or offset + byte_count > file_size:
                        raise ValueError(f"page {page_number} pixel data extends past end of file")
    except (tifffile.TiffFileError, OSError) as exc:
        raise ValueError(f"cannot read TIFF structure: {exc}") from exc


def validate_tiffs(paths: list[Path]) -> list[str]:
    errors = []
    for path in paths:
        try:
            validate_tiff(path)
        except ValueError as exc:
            errors.append(f"{path.name}: {exc}")
    return errors


def discover_experiment_folders(roots: list[Path]) -> list[Path]:
    folders = []
    for root in roots:
        root = Path(root)
        if not root.exists():
            print(f"!! root not found: {root}")
            continue

        for folder in root.rglob("*"):
            if folder.is_dir() and find_tiffs(folder):
                folders.append(folder)
    return folders


def deep_merge(base: dict, override: dict) -> dict:
    """Recursively merge ``override`` into ``base`` in place."""
    for key, value in override.items():
        if isinstance(value, dict) and isinstance(base.get(key), dict):
            deep_merge(base[key], value)
        else:
            base[key] = value
    return base


def load_native_settings() -> dict:
    """Load the GUI's native nested settings, merged onto Suite2p defaults."""
    settings = default_settings()
    if GUI_SETTINGS_PATH.exists():
        try:
            saved = np.load(GUI_SETTINGS_PATH, allow_pickle=True).item()
            deep_merge(settings, saved)
            print(f"Loaded GUI settings from {GUI_SETTINGS_PATH}")
        except Exception as exc:
            print(f"!! failed to load GUI settings, using defaults: {exc}")
    else:
        print(f"GUI settings not found at {GUI_SETTINGS_PATH}; using defaults")
    return settings


def apply_overrides(settings: dict, overrides: dict | None) -> None:
    """Route each OVERRIDES entry to its location in the nested settings dict.

    Keys may be a flat name from SETTINGS_KEY_PATHS (e.g. "fs", "npix_norm_max")
    or an explicit dotted path (e.g. "detection.block_size").
    """
    for key, value in (overrides or {}).items():
        if value is None:
            continue
        if "." in key:
            path = tuple(key.split("."))
        else:
            path = SETTINGS_KEY_PATHS.get(key)
        if path is None:
            print(f"!! Unknown override key ignored: {key}")
            continue
        target = settings
        for part in path[:-1]:
            target = target[part]
        target[path[-1]] = value
        print(f"Override: {'.'.join(path)} = {value}")


def coerce_settings_types(settings: dict, schema: dict = SETTINGS) -> None:
    """Cast legacy float values to the bool/int types Suite2p's schema declares.

    Old GUI ops store many flags as floats (e.g. two_step_registration=0.0),
    which breaks code paths that expect ints/bools (e.g. range(...)).
    """
    for key, spec in schema.items():
        if key not in settings:
            continue
        if "description" in spec:  # leaf parameter
            value = settings[key]
            if value is None:
                continue
            caster = spec.get("type")
            if caster is bool:
                settings[key] = bool(value)
            elif caster is int and isinstance(value, float) and float(value).is_integer():
                settings[key] = int(value)
            elif caster is list and isinstance(value, (list, tuple)):
                settings[key] = [
                    int(v) if isinstance(v, float) and float(v).is_integer() else v
                    for v in value
                ]
        elif isinstance(settings.get(key), dict):  # nested section
            coerce_settings_types(settings[key], spec)


def build_db_and_settings(folder: Path, overrides: dict | None = None) -> tuple[dict, dict]:
    settings = load_native_settings()
    apply_overrides(settings, overrides)
    coerce_settings_types(settings)

    if FORCE_GPU:
        settings["torch_device"] = "cuda"
    settings["run"]["do_registration"] = int(RUN_REGISTRATION)

    # Batch behaviour: don't write registered TIFFs, delete the binary afterwards.
    settings["registration"]["reg_tif"] = False
    settings["io"]["delete_bin"] = True

    # Cellpose needs a positive, equal-aspect diameter (0 -> 0/0 division in roi_detect).
    diam = settings.get("diameter")
    if not isinstance(diam, (list, tuple, np.ndarray)):
        diam = [diam, diam]
    diam = [float(d) for d in diam]
    if not all(d > 0 for d in diam):
        diam = [float(CELL_DIAMETER), float(CELL_DIAMETER)]
    settings["diameter"] = diam

    db = default_db()
    db["data_path"] = [str(folder)]
    db["save_path0"] = str(folder)
    db["fast_disk"] = str(folder)
    db["input_format"] = "tif"
    db["nchannels"] = NCHANNELS
    db["functional_chan"] = FUNCTIONAL_CHAN
    db["look_one_level_down"] = False
    db["subfolders"] = None

    return db, settings


def run_suite2p_on_folder(
    folder: Path,
    overrides: dict | None = None,
    fresh_reprocess: bool = False,
) -> None:
    print(f"\n=== Running Suite2p on: {folder} ===")

    removed_cache = remove_incomplete_cache(folder, force=fresh_reprocess)
    if removed_cache:
        print("Removed stale Suite2p outputs:")
        for path in removed_cache:
            print(f" - {path.name}")

    db, settings = build_db_and_settings(folder, overrides=overrides)
    run_s2p(db=db, settings=settings)


def batch_s2p(
    roots: list[Path],
    overrides: dict | None = None,
    skip_if_processed: bool = True,
    fresh_reprocess: bool = False,
) -> list[dict]:
    patch_suite2p_tiff_reader()
    folders = discover_experiment_folders(roots)
    print(f"Found {len(folders)} folder(s) with TIFF files")

    results = []
    from tqdm import tqdm
    for folder in tqdm(folders, desc="Processing folders"):
        if skip_if_processed and is_already_processed(folder):
            print(f"-> Skipping already processed folder: {folder}")
            results.append({"folder": folder, "status": "skipped_processed"})
            continue

        tiff_errors = validate_tiffs(find_tiffs(folder))
        if tiff_errors:
            print(f"!! Skipping invalid TIFF folder: {folder}")
            for error in tiff_errors:
                print(f"   - {error}")
            results.append({"folder": folder, "status": "skipped_invalid_tiff", "errors": tiff_errors})
            continue

        try:
            run_suite2p_on_folder(
                folder, overrides=overrides, fresh_reprocess=fresh_reprocess
            )
            results.append({"folder": folder, "status": "processed"})
        except Exception as exc:
            print(f"!! Failed to process folder: {folder}")
            print(f"   - {type(exc).__name__}: {exc}")
            results.append({"folder": folder, "status": "failed", "error": str(exc)})

    return results


## Preview folders

Run this cell before launching the batch to confirm which folders will be processed.

In [5]:
folders = discover_experiment_folders(ROOTS)
print(f"Found {len(folders)} folder(s) with TIFF files")
for folder in folders:
    status = "processed" if is_already_processed(folder) else "pending"
    print(f"[{status}] {folder}")


Found 100 folder(s) with TIFF files
[pending] C:\Users\mzinn1\Desktop\DoD WT Cohorts\Ex Vivo Ca Imaging 710\DoD_ChABC_CTRL_Cohort 1\5965\5965Lepi-1
[pending] C:\Users\mzinn1\Desktop\DoD WT Cohorts\Ex Vivo Ca Imaging 710\DoD_ChABC_CTRL_Cohort 1\5965\5965Repi-1
[pending] C:\Users\mzinn1\Desktop\DoD WT Cohorts\Ex Vivo Ca Imaging 710\DoD_ChABC_CTRL_Cohort 1\5965\5968Lepi-1
[pending] C:\Users\mzinn1\Desktop\DoD WT Cohorts\Ex Vivo Ca Imaging 710\DoD_ChABC_CTRL_Cohort 1\5971\5971Lepi-1
[pending] C:\Users\mzinn1\Desktop\DoD WT Cohorts\Ex Vivo Ca Imaging 710\DoD_ChABC_CTRL_Cohort 1\5971\5971Lepi-2
[processed] C:\Users\mzinn1\Desktop\DoD WT Cohorts\Ex Vivo Ca Imaging 710\DoD_ChABC_CTRL_Cohort 1\5971\5971Lepi-3
[pending] C:\Users\mzinn1\Desktop\DoD WT Cohorts\Ex Vivo Ca Imaging 710\DoD_ChABC_CTRL_Cohort 1\5971\5971Repi-1
[pending] C:\Users\mzinn1\Desktop\DoD WT Cohorts\Ex Vivo Ca Imaging 710\DoD_ChABC_CTRL_Cohort 1\5972\5972Repi-1
[pending] C:\Users\mzinn1\Desktop\DoD WT Cohorts\Ex Vivo Ca Imagin

## Run batch processing

In [6]:
results = batch_s2p(
    ROOTS,
    overrides=OVERRIDES,
    skip_if_processed=False,
    fresh_reprocess=FRESH_REPROCESS,
)

print("\nDone.")
for result in results:
    print(f"{result['status']}: {result['folder']}")


Found 100 folder(s) with TIFF files


Processing folders:   0%|          | 0/100 [00:00<?, ?it/s]


=== Running Suite2p on: C:\Users\mzinn1\Desktop\DoD WT Cohorts\Ex Vivo Ca Imaging 710\DoD_ChABC_CTRL_Cohort 1\5965\5965Lepi-1 ===
Removed stale Suite2p outputs:
 - data.bin
 - data_chan2.bin
 - db.npy
 - settings.npy
 - reg_outputs.npy
Loaded GUI settings from C:\Users\mzinn1\.suite2p\settings\settings_user.npy
Override: fs = 3.0
Override: detection.algorithm = cellpose
Override: detection.cellpose_settings.cellpose_model = cpdino_ExVivoGCaMP


c:\Users\mzinn1\AppData\Local\anaconda3\envs\gcamp\lib\site-packages\numpy\_core\fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
c:\Users\mzinn1\AppData\Local\anaconda3\envs\gcamp\lib\site-packages\numpy\_core\_methods.py:145: RuntimeWarning: invalid value encountered in divide
  ret = ret.dtype.type(ret / rcount)
Processing folders:   1%|          | 1/100 [00:05<08:20,  5.06s/it]

!! Failed to process folder: C:\Users\mzinn1\Desktop\DoD WT Cohorts\Ex Vivo Ca Imaging 710\DoD_ChABC_CTRL_Cohort 1\5965\5965Lepi-1
   - ValueError: no ROIs were found -- check registered binary and maybe try changing spatial scale / diameter / threshold_scaling

=== Running Suite2p on: C:\Users\mzinn1\Desktop\DoD WT Cohorts\Ex Vivo Ca Imaging 710\DoD_ChABC_CTRL_Cohort 1\5965\5965Repi-1 ===
Removed stale Suite2p outputs:
 - data.bin
 - data_chan2.bin
 - db.npy
 - settings.npy
 - reg_outputs.npy
Loaded GUI settings from C:\Users\mzinn1\.suite2p\settings\settings_user.npy
Override: fs = 3.0
Override: detection.algorithm = cellpose
Override: detection.cellpose_settings.cellpose_model = cpdino_ExVivoGCaMP


...

!! Failed to process folder: C:\Users\mzinn1\Desktop\DoD WT Cohorts\Ex Vivo Ca Imaging 710\DoD_TSP-1_ONC_Cohort 1\5840\5840L-1
   - ValueError: no ROIs were found -- check registered binary and maybe try changing spatial scale / diameter / threshold_scaling

=== Running Suite2p on: C:\Users\mzinn1\Desktop\DoD WT Cohorts\Ex Vivo Ca Imaging 710\DoD_TSP-1_ONC_Cohort 1\6040\6040R-1 ===
Loaded GUI settings from C:\Users\mzinn1\.suite2p\settings\settings_user.npy
Override: fs = 3.0
Override: detection.algorithm = cellpose
Override: detection.cellpose_settings.cellpose_model = cpdino_ExVivoGCaMP


Processing folders:  99%|█████████▉| 99/100 [07:53<00:02,  2.72s/it]


=== Running Suite2p on: C:\Users\mzinn1\Desktop\DoD WT Cohorts\Ex Vivo Ca Imaging 710\DoD_TSP-1_ONC_Cohort 1\6040\6040R-2 ===
Loaded GUI settings from C:\Users\mzinn1\.suite2p\settings\settings_user.npy
Override: fs = 3.0
Override: detection.algorithm = cellpose
Override: detection.cellpose_settings.cellpose_model = cpdino_ExVivoGCaMP


Processing folders: 100%|██████████| 100/100 [07:59<00:00,  4.79s/it]


Done.
failed: C:\Users\mzinn1\Desktop\DoD WT Cohorts\Ex Vivo Ca Imaging 710\DoD_ChABC_CTRL_Cohort 1\5965\5965Lepi-1
failed: C:\Users\mzinn1\Desktop\DoD WT Cohorts\Ex Vivo Ca Imaging 710\DoD_ChABC_CTRL_Cohort 1\5965\5965Repi-1
failed: C:\Users\mzinn1\Desktop\DoD WT Cohorts\Ex Vivo Ca Imaging 710\DoD_ChABC_CTRL_Cohort 1\5965\5968Lepi-1
failed: C:\Users\mzinn1\Desktop\DoD WT Cohorts\Ex Vivo Ca Imaging 710\DoD_ChABC_CTRL_Cohort 1\5971\5971Lepi-1
failed: C:\Users\mzinn1\Desktop\DoD WT Cohorts\Ex Vivo Ca Imaging 710\DoD_ChABC_CTRL_Cohort 1\5971\5971Lepi-2
processed: C:\Users\mzinn1\Desktop\DoD WT Cohorts\Ex Vivo Ca Imaging 710\DoD_ChABC_CTRL_Cohort 1\5971\5971Lepi-3
failed: C:\Users\mzinn1\Desktop\DoD WT Cohorts\Ex Vivo Ca Imaging 710\DoD_ChABC_CTRL_Cohort 1\5971\5971Repi-1
failed: C:\Users\mzinn1\Desktop\DoD WT Cohorts\Ex Vivo Ca Imaging 710\DoD_ChABC_CTRL_Cohort 1\5972\5972Repi-1
failed: C:\Users\mzinn1\Desktop\DoD WT Cohorts\Ex Vivo Ca Imaging 710\DoD_ChABC_CTRL_Cohort 1\5972\5972Repi-2


## Check outputs

After processing, this cell summarizes whether each folder has the expected `plane0` output files.

In [8]:
for folder in discover_experiment_folders(ROOTS):
    plane0 = folder / "suite2p" / "plane0"
    outputs = {name: (plane0 / name).is_file() for name in ("F.npy", "Fneu.npy", "spks.npy", "iscell.npy", "ops.npy")}
    print(folder)
    for name, exists in outputs.items():
        print(f"  {name}: {'OK' if exists else 'missing'}")


C:\Users\mzinn1\Desktop\GCaMP6s_EX37x_Days_Repeating\BP\1-1
  F.npy: OK
  Fneu.npy: OK
  spks.npy: OK
  iscell.npy: OK
  ops.npy: OK
C:\Users\mzinn1\Desktop\GCaMP6s_EX37x_Days_Repeating\BP\1-1_Day10
  F.npy: OK
  Fneu.npy: OK
  spks.npy: OK
  iscell.npy: OK
  ops.npy: OK
C:\Users\mzinn1\Desktop\GCaMP6s_EX37x_Days_Repeating\BP\1-1_Day2
  F.npy: OK
  Fneu.npy: OK
  spks.npy: OK
  iscell.npy: OK
  ops.npy: OK
C:\Users\mzinn1\Desktop\GCaMP6s_EX37x_Days_Repeating\BP\1-1_Day3
  F.npy: OK
  Fneu.npy: OK
  spks.npy: OK
  iscell.npy: OK
  ops.npy: OK
C:\Users\mzinn1\Desktop\GCaMP6s_EX37x_Days_Repeating\BP\1-1_Day4
  F.npy: OK
  Fneu.npy: OK
  spks.npy: OK
  iscell.npy: OK
  ops.npy: OK
C:\Users\mzinn1\Desktop\GCaMP6s_EX37x_Days_Repeating\BP\1-1_Day5
  F.npy: OK
  Fneu.npy: OK
  spks.npy: OK
  iscell.npy: OK
  ops.npy: OK
C:\Users\mzinn1\Desktop\GCaMP6s_EX37x_Days_Repeating\BP\1-1_Day7
  F.npy: OK
  Fneu.npy: OK
  spks.npy: OK
  iscell.npy: OK
  ops.npy: OK
C:\Users\mzinn1\Desktop\GCaMP6s_EX37x_